# ODAC25 target-bearing shard profile
Streams the first `val/mof_plus_adsorbate` database without storing the full archive, then records target availability. CPU is sufficient.

In [ ]:
%pip install -q "huggingface-hub>=0.28" "ase>=3.26" "fairchem-core>=2,<3" "fairchem-data-odac>=0.1,<1"

In [ ]:
import json, os, subprocess, sys, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
hits = list(Path('/kaggle/input').rglob('mof_dac/odac25_archive.py'))
assert len(hits) == 1, f'Attach one current MOF_DAC_Kaggle_bundle Dataset: {hits}'
REPO = hits[0].parents[1]
os.chdir(REPO)
OUT = Path('/kaggle/working/odac25_target')
OUT.mkdir(parents=True, exist_ok=True)
print('repository:', REPO)

The stream may traverse much of the 12 GB compressed validation archive, but only one database is retained. Limits stop an unexpected archive layout.

In [ ]:
VAL_URL = 'https://dl.fbaipublicfiles.com/dac/odac25/20250918/odac25_filtered_val.tar.gz'
subprocess.run([sys.executable, '-m', 'mof_dac.odac25_archive', VAL_URL,
                '--prefix', 'val/mof_plus_adsorbate',
                '--output', str(OUT/'data'),
                '--manifest', str(OUT/'archive_member.json'),
                '--max-scan-gb', '20', '--max-member-gb', '10'], check=True)
manifest = json.loads((OUT/'archive_member.json').read_text())
DATABASE = Path(manifest['selected']['local_path'])
DATASET_SOURCE = DATABASE.parent
print(json.dumps(manifest, indent=2))

In [ ]:
subprocess.run([sys.executable, 'scripts/odac25_hf.py', 'inspect', str(DATASET_SOURCE),
                '--output', str(OUT/'target_schema.json'), '--samples', '32'], check=True)
subprocess.run([sys.executable, 'scripts/odac25_hf.py', 'profile-targets', str(DATASET_SOURCE),
                '--output', str(OUT/'target_profile.json'), '--split', 'val'], check=True)
subprocess.run([sys.executable, 'scripts/odac25_hf.py', 'materialize-targets', str(DATASET_SOURCE),
                '--output', str(OUT/'paired_targets.json'), '--split', 'val'], check=True)
print((OUT/'target_profile.json').read_text())
print((OUT/'paired_targets.json').read_text()[:12000])

The profile exposes target availability. Materialization then applies the frozen rule: exact one-molecule CO2/H2O rows, largest `fid` per trajectory, and lowest final-frame corrected adsorption energy per MOF/gas. This is a strongest-sampled-configuration diagnostic, not equilibrium selectivity.

In [ ]:
deliverables = ['archive_member.json', 'target_schema.json', 'target_profile.json', 'paired_targets.json']
archive = Path('/kaggle/working/ODAC25_target_analysis_artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in deliverables:
        assert (OUT/name).is_file(), name
        zf.write(OUT/name, arcname=name)
print('Download from Notebook Output:', archive)